In [1]:
import os 
import sys 
import warnings
sys.path.append(os.path.abspath(os.path.join("..")))

from src.spark import get_spark_session
from configs.config import SILVER_DATA_DIR, GOLD_DATA_DIR

warnings.filterwarnings("ignore")

In [ ]:
spark = get_spark_session()

# `READ DATA FROM SILVER`

In [3]:
silver_path = SILVER_DATA_DIR / "youtube_channels"

df_gold = spark.read.parquet(str(silver_path))

In [4]:
df_gold.show()

+----+--------------------+-----------+-----------+-----------+--------------------+-------+--------------------+--------------------+
|rank|            youtuber|subscribers|video_views|video_count|            category|started|views_per_subscriber| avg_views_per_video|
+----+--------------------+-----------+-----------+-----------+--------------------+-------+--------------------+--------------------+
|   1|        Tsuriki Show|   34100000|42490526838|       4739|       Entertainment|   2019|  1246.0565055131965|    8966137.75859886|
|   2|Kidibli (kinder S...|   29600000|15673364837|       1236|       Entertainment|   2015|   529.5055688175676|1.2680715887540452E7|
|   3|Kurzgesagt – In A...|   23600000| 3145706013|        271|           Education|   2013|  133.29262766949154|1.1607771265682656E7|
|   4|            Boxtoxtv|   23500000|18303986629|       1559|              Comedy|   2022|   778.8930480425532|1.1740850948685054E7|
|   5|          Haertetest|   19500000| 3420864412|    

In [5]:
df_gold.printSchema()

root
 |-- rank: integer (nullable = true)
 |-- youtuber: string (nullable = true)
 |-- subscribers: long (nullable = true)
 |-- video_views: long (nullable = true)
 |-- video_count: long (nullable = true)
 |-- category: string (nullable = true)
 |-- started: integer (nullable = true)
 |-- views_per_subscriber: double (nullable = true)
 |-- avg_views_per_video: double (nullable = true)



In [6]:
df_gold.createOrReplaceTempView("df_gold")

### `CHANNEL PERFORMANCE`

In [7]:
query = """
    SELECT 
        rank,
        youtuber,
        subscribers,
        video_views,
        video_count,
        category,
        started
    FROM df_gold
"""

gold_channel_performance = spark.sql(query)

In [8]:
gold_channel_path = GOLD_DATA_DIR / "channel_performance"

(
    gold_channel_performance.write.mode("overwrite").parquet(str(gold_channel_path))
)

### `CATEGORY PERFORMANCE`

In [9]:
query = """
    SELECT 
        category,
        AVG(video_views) as avg_views_per_category,
        SUM(video_count) as total_video_count_per_category
    FROM df_gold
    GROUP BY category
"""


category_performance = spark.sql(query)

In [10]:
gold_category_path = GOLD_DATA_DIR / "category performance"

(
    category_performance.write.mode("overwrite").parquet(str(gold_category_path))
)

In [11]:
channel_gold = spark.read.parquet(str(gold_channel_path))

In [12]:
channel_gold.show(10)

+----+--------------------+-----------+-----------+-----------+--------------------+-------+
|rank|            youtuber|subscribers|video_views|video_count|            category|started|
+----+--------------------+-----------+-----------+-----------+--------------------+-------+
|   1|        Tsuriki Show|   34100000|42490526838|       4739|       Entertainment|   2019|
|   2|Kidibli (kinder S...|   29600000|15673364837|       1236|       Entertainment|   2015|
|   3|Kurzgesagt – In A...|   23600000| 3145706013|        271|           Education|   2013|
|   4|            Boxtoxtv|   23500000|18303986629|       1559|              Comedy|   2022|
|   5|          Haertetest|   19500000| 3420864412|       1712|Science & Technology|   2011|
|   6|       Noel Robinson|   18600000|10752582783|       1639|       Entertainment|   2015|
|   7|        Family Booms|   16800000|16225259066|       1587|       Entertainment|   2021|
|   8|      Talking Angela|   13300000| 4116834514|        345|       

In [13]:
category_gold = spark.read.parquet(str(gold_category_path))

In [14]:
category_gold.show()

+--------------------+----------------------+------------------------------+
|            category|avg_views_per_category|total_video_count_per_category|
+--------------------+----------------------+------------------------------+
|      People & Blogs|   5.084452563333333E8|                        156665|
|       Entertainment|  1.0174514819525547E9|                        382548|
|              Gaming|   6.653264513354839E8|                        384064|
|               Music|   9.551903534655173E8|                         62630|
|      Pets & Animals|        1.3225049996E9|                         65556|
|     Travel & Events|          3.99934162E8|                          7752|
|              Comedy|        1.6753232596E9|                         23010|
|    Autos & Vehicles|   5.265360451052632E8|                         22759|
|Https://us.youtub...|   3.978006218787879E8|                         22007|
|           Education|  4.1612161489361703E8|                         54757|